# Correct the coherence for finite-look bias

NISAR coherence is estimated from a finite number of looks, which biases it high where the true coherence is low (Bamler and Hartl, 1998). This shows how the number of looks is found over open water, where the true coherence is ~0, and how the expected-coherence equation is inverted to get the true coherence for the 80 m and 20 m grids. The method follows Hoppinen et al. (2026), Sec. 3.6.

In [ ]:
from shapely.geometry import box
import geopandas as gpd
import numpy as np
import xarray as xr
import rioxarray  # noqa: F401
from rasterio.enums import Resampling
import matplotlib.pyplot as plt
from seaborn import set_theme
set_theme()

import sys
import os
from pathlib import Path

# import from this repo 
project_root = str(Path(os.getcwd()).parent)
if project_root not in sys.path:
    sys.path.append(project_root)

%load_ext autoreload
%autoreload 2
from scripts.download import download_nisar, build_cube
from scripts.coherence import expected_coherence, debias_coherence, effective_looks_factor

First, we setup logging to get useful information and error messages. 

In [ ]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    force=True 
)

## Download NISAR data over water

The true coherence over a lake or reservoir is assumed to be zero, so any coherence there is bias. Fill in the track, frame, dates and bounding box of a water body; use the same track and frame as the study area so the multilooking matches.

In [ ]:
track = None   # TODO
frame = None   # TODO
start = None   # TODO e.g. '2025-12-01'
end = None     # TODO e.g. '2026-04-30'
proc_code = None   # TODO e.g. 'P05023'

# TODO: bounding box of the water body, in the crs below
bounds = (None, None, None, None)   # (minx, miny, maxx, maxy)
crs = None   # TODO e.g. 'EPSG:32613'
shp = gpd.GeoDataFrame(geometry=[box(*bounds)], crs=crs)

data_dir = "../data/"
out = data_dir + f"nisar/water/track{str(track).zfill(3)}_frame{str(frame).zfill(3)}"
layers = ['unwrapped_phase', 'coherence_unwrapped',  # 80 m interferogram 
          'wrapped_phase', 'coherence_wrapped']      # 20 m interferogram 

Download the clipped coherence layers. Files that already exist are skipped.

In [ ]:
(grans, files) = download_nisar(
    track=track,
    frame=frame,
    start_date=start,
    end_date=end,
    aoi=shp,
    layers=layers,
    output_dir=out,
    crid=proc_code
)

Build a data cube for each grid, then plot the coherence over the water.

In [ ]:
wrapped_files = [f for f in files if "coherence_wrapped" in str(f)]
unwrapped_files = [f for f in files if "coherence_unwrapped" in str(f)]

water_20m = build_cube(wrapped_files, "coherence", utc_offset_hours=-7)     # 20 m grid
water_80m = build_cube(unwrapped_files, "coherence", utc_offset_hours=-7)  # 80 m grid
water_20m = water_20m.rio.write_crs(water_20m.attrs['crs'])
water_80m = water_80m.rio.write_crs(water_80m.attrs['crs'])

water_80m['coherence'].plot(col='pair', col_wrap=5, cmap='inferno', vmin=0, vmax=1)

## True looks

NISAR multilooks the interferogram 5x6 for the 20 m layer and 13x16 for the 80 m layer, so the nominal looks are N = 30 and 208. Geocoding resamples these pixels, so neighbors are partly correlated and the effective looks are L = N / f, where f is an oversampling factor.

In [ ]:
n_20 = 5 * 6
n_80 = 13 * 16

Over water the true coherence is zero at both resolutions, so f is the value for which the debiased 20 m and 80 m coherences agree. Water pixels are those with 80 m coherence below 0.2, which removes coherent shoreline pixels, and the 20 m coherence is block-averaged onto the 80 m grid so both are sampled on the same footprint.

In [ ]:
coh_80 = water_80m['coherence']
coh_20 = water_20m['coherence'].rio.reproject_match(water_80m, resampling=Resampling.average)

is_water = coh_80 < 0.2
coh_80_water = coh_80.where(is_water).mean().item()
coh_20_water = coh_20.where(is_water).mean().item()
print(f"mean water coherence: 20 m = {coh_20_water:.3f}, 80 m = {coh_80_water:.3f}")

In [ ]:
f = effective_looks_factor(coh_20_water, coh_80_water, n_20, n_80)
L_20 = n_20 / f
L_80 = n_80 / f
print(f"f = {f:.2f}, L_20 = {L_20:.0f}, L_80 = {L_80:.0f}")

## Invert the Bamler equation

Eq. (62) of Bamler and Hartl (1998) gives the expected sample coherence for a true coherence and number of looks. It overestimates low coherence and approaches the 1:1 line as the looks increase. Inverting it, which `debias_coherence` does by interpolation, gives the true coherence from the observed coherence.

In [ ]:
true = np.linspace(0, 0.98, 100)

fig, ax = plt.subplots(figsize=(5, 5))
for L, label in [(L_20, '20 m'), (L_80, '80 m')]:
    ax.plot(true, expected_coherence(true, L), label=f'{label} (L = {L:.0f})')
ax.plot([0, 1], [0, 1], 'k--', label='1:1')
ax.set_xlabel('True coherence')
ax.set_ylabel('Expected sample coherence')
ax.legend()
plt.tight_layout()

Check the inversion by passing the expected coherence back through it, which should return the true coherence.

In [ ]:
check = debias_coherence(xr.DataArray(expected_coherence(true, L_80)), L_80)
print(f"max round-trip error: {np.abs(check.values - true).max():.1e}")

## True coherence for the NISAR cubes

Open the data cubes that were saved by the download notebook and debias the coherence of each grid with its own effective looks. The 20 m layer is debiased natively here; Hoppinen et al. (2026) block-average it to 80 m first, which leaves less residual bias.

In [ ]:
zarr_dir = data_dir + f'nisar/zarrs/track{str(track).zfill(3)}_frame{str(frame).zfill(3)}/'
nisar_20m = xr.open_zarr(zarr_dir + 'obs_20m.zarr')
nisar_80m = xr.open_zarr(zarr_dir + 'obs_80m.zarr')

In [ ]:
nisar_20m['coherence_true'] = debias_coherence(nisar_20m['coherence'], L_20)
nisar_80m['coherence_true'] = debias_coherence(nisar_80m['coherence'], L_80)

Plot the observed and true coherence of the first date pair to see the change.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for row, (ds, res) in enumerate([(nisar_80m, '80 m'), (nisar_20m, '20 m')]):
    for ax, var in zip(axes[row], ['coherence', 'coherence_true']):
        ds[var].isel(pair=0).squeeze().plot(ax=ax, cmap='inferno', vmin=0, vmax=1)
        ax.set_title(f'{res} {var}')
plt.tight_layout()

Save the true coherence back to the zarrs as a new variable, leaving the observed coherence unchanged.

In [ ]:
nisar_20m[['coherence_true']].to_zarr(zarr_dir + 'obs_20m.zarr', mode="a")
nisar_80m[['coherence_true']].to_zarr(zarr_dir + 'obs_80m.zarr', mode="a")